# JSON - Duplicates

Duplicate complete observations, either identical or with conflicting values in one measurement array.

**Instructions:** Use `prepare_data_json.ipynb` to create a clean `data/weather.json` first. This notebook overwrites it; restore the clean source before rerunning.

In [ ]:
from pathlib import Path
from random import Random
import copy
import json

import pandas as pd

# Configuration
DATA_PATH = Path("data/weather.json")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive, local calendar dates.
SEED = 814205

# Duplicates
DUPLICATE_KIND = "mixed"  # "exact", "conflicting", "mixed", "random".
DUPLICATE_PLACEMENT = "mixed"
DUPLICATE_RATE = (0.01, 0.04)  # Share of eligible original rows copied once.
DUPLICATE_GROUPS = 3
DUPLICATE_GROUPED_SHARE = 0.70
DUPLICATE_CONFLICT_COLUMN = "temperature_2m"
DUPLICATE_CONFLICT_OFFSET = 1.5
DUPLICATE_CONFLICT_SHARE = 0.50


In [ ]:
# Shared JSON helpers: include once in a combined dirty_data.py.
def eligible_positions(document, *, time_column, timezone, start_date, end_date):
    hourly = document.get("hourly")
    if not isinstance(hourly, dict) or time_column not in hourly:
        raise ValueError("Expected a source-shaped document with hourly arrays.")
    if not all(isinstance(v, list) for v in hourly.values()):
        raise ValueError("All hourly fields must contain arrays.")
    if len({len(v) for v in hourly.values()}) != 1:
        raise ValueError("Hourly arrays must be aligned before ordinary mutations.")
    times = hourly[time_column]
    if not times or any(isinstance(t, bool) or not isinstance(t, (int, float)) for t in times):
        raise ValueError("Expected nonempty numeric Unix-second timestamps.")
    if any(a >= b for a, b in zip(times, times[1:])):
        raise ValueError("Expected unique increasing timestamps; apply duplicates last.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC").timestamp()
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC").timestamp()
    if start >= end:
        raise ValueError("Start date must precede end date.")
    positions = [i for i, t in enumerate(times) if start <= t < end]
    if not positions:
        raise ValueError("No observations in the selected interval.")
    return positions, start, end


def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_duplicates_json(
    document, rng, *, time_column, timezone, start_date, end_date,
    duplicate_kind, placement, duplicate_rate, n_groups,
    grouped_share, conflict_column, conflict_offset, conflict_share,
):
    positions, _, _ = eligible_positions(
        document, time_column=time_column, timezone=timezone,
        start_date=start_date, end_date=end_date,
    )
    if duplicate_kind not in ("exact", "conflicting", "mixed", "random"):
        raise ValueError("Unknown duplicate kind.")
    if not 0 < conflict_share < 1:
        raise ValueError("Invalid conflict share.")
    kind = rng.choice(("exact", "conflicting", "mixed")) if duplicate_kind == "random" else duplicate_kind
    hourly = document["hourly"]
    if kind != "exact":
        if conflict_column not in hourly or conflict_column == time_column or conflict_offset == 0:
            raise ValueError("Conflicting copies need a numeric non-key field and nonzero offset.")
        # In mixed exercises, don't try to create numeric conflicts from existing nulls.
        positions = [i for i in positions
                     if isinstance(hourly[conflict_column][i], (int, float))
                     and not isinstance(hourly[conflict_column][i], bool)]
    selected, _ = select_positions(
        len(positions), rng, method=placement, rate=duplicate_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    if kind == "mixed" and len(selected) < 2:
        raise ValueError("Mixed duplicate kinds need at least two copies.")
    count = (0 if kind == "exact" else len(selected) if kind == "conflicting" else
             max(1, min(len(selected)-1, round(len(selected) * conflict_share))))
    hourly = document["hourly"]
    conflict_source_positions = set()
    if count:
        if conflict_column not in hourly or conflict_column == time_column:
            raise ValueError("Conflict field must be an existing measurement array.")
        if conflict_offset == 0:
            raise ValueError("Conflict offset must be nonzero.")
        valid_sources = [positions[i] for i in selected
                         if isinstance(hourly[conflict_column][positions[i]], (int, float))
                         and not isinstance(hourly[conflict_column][positions[i]], bool)]
        if len(valid_sources) < count:
            raise ValueError("Not enough non-null numeric values for conflicting copies.")
        conflict_source_positions = set(rng.sample(valid_sources, k=count))
    # Each selected observation is copied immediately after its original.
    selected_source = {positions[i] for i in selected}
    dirty = copy.deepcopy(document)
    new_hourly = {}
    for name, values in hourly.items():
        result = []
        for i, value in enumerate(values):
            result.append(value)
            if i in selected_source:
                copy_value = value
                if i in conflict_source_positions and name == conflict_column:
                    copy_value = value + conflict_offset
                result.append(copy_value)
        new_hourly[name] = result
    dirty["hourly"] = new_hourly
    return dirty


In [ ]:
# Small, reproducible source-shaped trial (no network required).
times = pd.date_range("2025-08-01", periods=24*14, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = {
    "latitude": 51.04, "timezone": "GMT", "hourly_units": {"temperature_2m": "°C"},
    "hourly": {
        "time": [int(t.timestamp()) for t in times],
        "temperature_2m": [float(i % 24) for i in range(len(times))],
        "relative_humidity_2m": [float(40 + i % 50) for i in range(len(times))],
    },
}

trial_dirty = add_duplicates_json(
    trial, Random(SEED), time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date="2025-08-15",
    duplicate_kind=DUPLICATE_KIND, placement=DUPLICATE_PLACEMENT,
    duplicate_rate=DUPLICATE_RATE, n_groups=DUPLICATE_GROUPS,
    grouped_share=DUPLICATE_GROUPED_SHARE,
    conflict_column=DUPLICATE_CONFLICT_COLUMN, conflict_offset=DUPLICATE_CONFLICT_OFFSET,
    conflict_share=DUPLICATE_CONFLICT_SHARE,
)
assert trial_dirty["latitude"] == trial["latitude"]
assert len({len(v) for v in trial_dirty["hourly"].values()}) == 1
assert len(trial_dirty["hourly"][TIME_COLUMN]) > len(trial["hourly"][TIME_COLUMN])
trial_dirty["hourly"][TIME_COLUMN][:8]

In [ ]:
# Apply to the prepared JSON (this overwrites weather.json).
if DATA_PATH.exists():
    with DATA_PATH.open("r", encoding="utf-8") as f:
        clean = json.load(f)
    rng = Random(SEED)
    dirty = add_duplicates_json(
        clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
        start_date=START_DATE, end_date=END_DATE,
        duplicate_kind=DUPLICATE_KIND, placement=DUPLICATE_PLACEMENT,
        duplicate_rate=DUPLICATE_RATE, n_groups=DUPLICATE_GROUPS,
        grouped_share=DUPLICATE_GROUPED_SHARE,
        conflict_column=DUPLICATE_CONFLICT_COLUMN, conflict_offset=DUPLICATE_CONFLICT_OFFSET,
        conflict_share=DUPLICATE_CONFLICT_SHARE,
    )
    serialized = json.dumps(dirty, allow_nan=False)
    DATA_PATH.write_text(serialized, encoding="utf-8")
    saved = json.loads(DATA_PATH.read_text(encoding="utf-8"))
else:
    clean, dirty, saved = trial, trial_dirty, trial_dirty
    print("Prepared data/weather.json not found: showing controlled trial only.")


In [ ]:
# Inspect the document as an analytical table.
df = pd.DataFrame(saved["hourly"])
print("Shape:", df.shape)
print("Any nulls:", df.isna().any().any())
print("Nulls by field:", df.isna().sum().to_dict())
print("Duplicate times:", df[TIME_COLUMN].duplicated().sum())
df.head()

In [ ]:
# Developer-side verification against the known clean document.
assert {k: v for k, v in saved.items() if k != "hourly"} == {k: v for k, v in clean.items() if k != "hourly"}
assert len({len(v) for v in saved["hourly"].values()}) == 1
old_times, new_times = clean["hourly"][TIME_COLUMN], saved["hourly"][TIME_COLUMN]
assert len(new_times) > len(old_times)
assert set(old_times) == set(new_times)
assert new_times == sorted(new_times)
assert new_times.count(new_times[0]) >= 1
print("Extra observation copies:", len(new_times) - len(old_times))


In [ ]:
# Developer diagnostic: additional copies by local date.
counts = pd.Series(saved["hourly"][TIME_COLUMN]).value_counts()
repeated = counts.loc[counts.gt(1)] - 1
local = pd.to_datetime(repeated.index.to_numpy(), unit="s", utc=True).tz_convert(TIMEZONE)
per_day = pd.Series(repeated.values, index=local).groupby(local.date).sum()
ax = per_day.plot(kind="bar", figsize=(10, 3), title="Duplicate copies by local date")
ax.set_xlabel("Local date")
ax.set_ylabel("Extra copies")
